# A real case study: time-resolved fluorescence 📁

The first two notebooks used clean, simulated data so you could learn
the cycle. This notebook runs that **same cycle on a real, published
measurement**: time-resolved fluorescence (emission) from
**Photosystem I**, a classic dataset in the field.

This is a simplified version of one of the official
[pyglotaran example case studies](https://github.com/glotaran/pyglotaran-examples).
Here we run only the **global analysis** with a sequential scheme; the
full study goes one step further with a **target analysis** (more on
that in notebook 05). The data file lives in
`04_fluorescence/data.ascii`.

Two things differ from the earlier notebooks:

1. The data comes straight from an instrument file (`.ascii`), which
   pyglotaran reads directly with `load_dataset` — no CSV helper needed.
2. The model has **four** states and a small streak-camera correction,
   because that is what this real measurement needs.

## Step 0: Load the tools we need

In [ ]:
from glotaran.io import load_dataset, load_model, load_parameters, save_result
from glotaran.optimization.optimize import optimize
from glotaran.project.scheme import Scheme

from pyglotaran_extras import plot_data_overview, plot_overview

## Step 1: Inspect the data

`load_dataset` reads the measurement file directly. pyglotaran reads
several common formats out of the box, including the plain-text
time-explicit `.ascii` format used here.

As always, start by inspecting the data overview to see how the
emission evolves in time before specifying a model.

> **Using your own data later?** If it is a CSV export, use the
> `load_csv_dataset` helper from notebooks 02 and 03. If it is already
> in a format pyglotaran supports (such as this `.ascii`), `load_dataset`
> reads it directly.

In [ ]:
data = load_dataset("04_fluorescence/data.ascii")

plot_data_overview(data, linlog=True, linthresh=1);

## Step 2: Specify the model and starting parameters

As before, the analysis recipe lives in two text files you can open
from the file browser:

- `04_fluorescence/model.yaml` — a four-state sequential (compartmental)
  kinetic scheme with a Gaussian instrument response function (this
  measurement was taken on a streak camera, so the model also includes
  a small "backsweep" correction).
- `04_fluorescence/parameters.yaml` — the starting guesses.

We load each piece explicitly here so you can see the parts that were
hidden inside `Scheme(...)` in the earlier notebooks.

In [ ]:
model = load_model("04_fluorescence/model.yaml")
parameters = load_parameters("04_fluorescence/parameters.yaml")

scheme = Scheme(
    model,
    parameters,
    data={"dataset1": data},
    maximum_number_function_evaluations=25,
)

scheme.validate()

## Step 3: Estimate the parameters (optimize)

Same as before: the optimizer refines the parameters to fit the data.

In [ ]:
result = optimize(scheme, raise_exception=True)
result

## Step 4: Validate and interpret the results

Check the residuals to confirm the four-state model describes the data,
then look at the four evolution-associated spectra and their
concentration profiles. Because this is emission analysed with a
sequential scheme, these are **evolution-associated spectra (EAS)** —
the emission counterpart of the EADS you saw with the simulated data.

In [ ]:
plot_overview(result.data["dataset1"], linlog=True, linthresh=10);

## (Optional) Save the results

In [ ]:
save_result(
    result=result,
    result_path="04_fluorescence/results/result.yml",
    allow_overwrite=True,
)
print("Saved! Look in the 04_fluorescence/results folder on the left.")

## 🚀 You have run a real case study

You now know the full cycle end to end — **inspect the data → specify a
model → estimate the parameters → validate and interpret** — and you
have run it on both simulated and real experimental data.

So far every model has been a **sequential** scheme, which gives
evolution-associated spectra. The natural next step is **target
analysis**: testing a specific physical kinetic scheme to recover the
true **species-associated spectra**.

Open the last notebook, **`05_going_further.ipynb`**, for where to go
next: target analysis, multi-dataset fits, and published case studies
to draw inspiration from.